# 第 33 课｜使用 PEFT 注入 LoRA，并验证每一条训练连接

**主线：先读模块 → 规划目标 → 注入 adapter → 审计参数 → 核对标签 → 检查梯度 → 保存、重载。**

前置：已经学过 LoRA 的 \(\Delta W=(\alpha/r)BA\)、冻结参数、自动求导、混合精度和真实模型模板。

本课不重复实现一个完整 PEFT，而是学习怎样**正确使用并验证一个真实库**。只对 Q/V 投影注入 LoRA；用少量短样本更新检查连接，不把它包装成微调效果。

材料：这份 Notebook 与同名 `.py` **都可以独立运行**。不需要前几课源码，不提供整套学习包。

## 0. 先看验证范围，避免混淆三个层次

| 层次 | 实际执行什么 | 是否下载权重 | 本讲义制作时的状态 |
|---|---|---|---|
| 离线数学／协议 | PyTorch 张量、标签、明确标注的接口替身 | 否 | 已执行 |
| 真实 PEFT 本地小模型 | 官方 Qwen3 类随机初始化＋真正的 `get_peft_model()` | 否 | 未执行：当前缺少 Transformers/PEFT |
| 真实预训练模型 | 固定 revision 的 Qwen3-0.6B＋真正的 PEFT | 是，或使用本地缓存 | 未执行；当前无 CUDA，也无可下载依赖的网络 |

**不能把第一行的成功当成后两行已通过。** 库导入失败、CUDA 不可用、网络失败都会明确保留；不会填入模拟模型输出。

默认“运行全部”只执行第一层。后两层需自行开启下面的开关。练习函数也需亲自填写，Notebook 全部执行完不等于完成练习。

真实阶段仍使用普通浮点基座，**不是 QLoRA**。不要直接拿第 31 课的量化对象或第 32 课的自定义 adapter 文件代入。

### 0.1 环境与公共工具

下一格是独立运行所需的定义，可折叠阅读。它不自动安装依赖或访问网络。环境报告显示实际版本，而不是声称一个预定版本已经安装。

In [1]:
from __future__ import annotations
import argparse
import contextlib
import copy
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import gc
import hashlib
import importlib.metadata
import json
import math
from pathlib import Path
import platform
import re
import sys
import tempfile
import traceback
from types import SimpleNamespace
from typing import Callable, Iterable

import torch
from torch import nn
from torch.nn import functional as F

MODEL_ID = "Qwen/Qwen3-0.6B"
REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
TARGETS = ("q_proj", "v_proj")
SEED = 3301
CONTRACT_FORMAT = "lesson33-adapter-contract-v1"
SOURCES = {
    "peft_lora": "https://huggingface.co/docs/peft/v0.18.0/package_reference/lora",
    "peft_model": "https://huggingface.co/docs/peft/v0.18.0/en/package_reference/peft_model",
    "peft_checkpoint": "https://huggingface.co/docs/peft/v0.18.0/en/developer_guides/checkpoint",
    "inplace_injection": "https://huggingface.co/docs/peft/guides/peft_model_config",
    "peft_mapping": "https://raw.githubusercontent.com/huggingface/peft/v0.18.0/src/peft/mapping_func.py",
    "qwen_source": "https://raw.githubusercontent.com/huggingface/transformers/v4.57.3/src/transformers/models/qwen3/modeling_qwen3.py",
    "causal_loss": "https://raw.githubusercontent.com/huggingface/transformers/v4.57.3/src/transformers/loss/loss_utils.py",
    "config": f"https://huggingface.co/{MODEL_ID}/blob/{REVISION}/config.json",
    "template": f"https://huggingface.co/{MODEL_ID}/raw/{REVISION}/tokenizer_config.json",
    "amp": "https://docs.pytorch.org/docs/stable/notes/amp_examples.html",
}
# 来自固定模型配置的课堂算例，不是本次读取真实权重得到的测量值。
CONFIG_EXAMPLE = dict(hidden_size=1024, num_hidden_layers=28, num_attention_heads=16,
                      num_key_value_heads=8, head_dim=128, vocab_size=151936)

@dataclass(frozen=True)
class Settings:
    rank: int = 8
    alpha: int = 16
    steps: int = 2
    lr: float = 1e-4
    max_length: int = 128
    device: str = "cpu"
    dtype: str = "auto"
    local_files_only: bool = False
    gradient_checkpointing: bool = False
    output_dir: str = "lesson_33_outputs"


def require(condition: bool, message: str) -> None:
    # 不依赖 assert，python -O 也不会关闭数据/接口检查。
    if not bool(condition):
        raise ValueError(message)


def positive_int(value, name: str) -> int:
    require(type(value) is int and value > 0, f"{name} 必须是正整数，不接受 bool")
    return value


def validate_settings(s: Settings) -> None:
    positive_int(s.rank, "rank")
    positive_int(s.alpha, "alpha")
    positive_int(s.max_length, "max_length")
    require(type(s.steps) is int and 0 <= s.steps <= 20, "本课 steps 限制为 0..20；不是正式训练器")
    require(not isinstance(s.lr, bool) and math.isfinite(s.lr) and s.lr > 0, "lr 必须有限且为正")
    require(s.device in {"cpu", "cuda", "auto"}, "device 应为 cpu/cuda/auto")
    require(s.dtype in {"auto", "float32", "float16", "bfloat16"}, "不支持的 dtype")


def json_write(path: Path, value) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def fingerprint(value) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                    separators=(",", ":")).encode()).hexdigest()


def environment() -> dict:
    versions = {}
    for pkg in ["torch", "transformers", "peft", "accelerate", "safetensors"]:
        try:
            versions[pkg] = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            versions[pkg] = None
    return {"python": platform.python_version(), "platform": platform.platform(),
            "packages": versions, "cuda_available": torch.cuda.is_available(),
            "cuda_build": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}


def new_run(root: str, kind: str) -> Path:
    parent = Path(root).resolve()
    parent.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    directory = parent / f"{kind}_{stamp}"
    directory.mkdir(exist_ok=False)
    return directory


def require_hf(*, need_peft: bool = True):
    try:
        import transformers
        from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
        from transformers import Qwen3Config, Qwen3ForCausalLM
        if need_peft:
            import peft
            from peft import LoraConfig, TaskType, get_peft_model, PeftModel
        else:
            peft = LoraConfig = TaskType = get_peft_model = PeftModel = None
    except (ImportError, RuntimeError) as exc:
        raise RuntimeError(
            "真实依赖未就绪；这不是模型测试通过。课程按 transformers 4.57.3、peft 0.18.0 接口编写。\n"
            '独立环境可安装：python -m pip install "transformers==4.57.3" "peft==0.18.0" '
            '"accelerate==1.12.0" safetensors\n'
            "当前环境若有依赖冲突，请保留完整原始异常。"
        ) from exc
    return SimpleNamespace(**locals())


def resolve_runtime(s: Settings):
    validate_settings(s)
    device = s.device
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("显式请求 CUDA，但不可用；不会静默回退")
    dtype_name = ("float16" if device == "cuda" else "float32") if s.dtype == "auto" else s.dtype
    require(device != "cpu" or dtype_name == "float32", "本课 CPU 路径固定 FP32，以简化验证")
    if dtype_name == "bfloat16" and not torch.cuda.is_bf16_supported():
        raise RuntimeError("当前 CUDA 环境不报告 BF16 支持；请用 float16/float32")
    return torch.device(device), getattr(torch, dtype_name)


def amp_context(device: torch.device, dtype: torch.dtype):
    if device.type == "cuda" and dtype in {torch.float16, torch.bfloat16}:
        return torch.autocast(device_type="cuda", dtype=dtype)
    return contextlib.nullcontext()

In [2]:
torch.set_num_threads(1)
print(json.dumps(environment(), ensure_ascii=False, indent=2))
print("默认不下载、不加载真实权重。")

{
  "python": "3.13.5",
  "platform": "Linux-6.18.57-x86_64-with-glibc2.41",
  "packages": {
    "torch": "2.10.0+cpu",
    "transformers": null,
    "peft": null,
    "accelerate": null,
    "safetensors": "0.7.0"
  },
  "cuda_available": false,
  "cuda_build": null,
  "gpu": null
}
默认不下载、不加载真实权重。


## 1. 从上一课的公式，走到 PEFT 的配置

\[
y=W_0x+\frac{\alpha}{r}BAx
\]

`get_peft_model()` 接收基座与 `LoraConfig`，替换目标模块并注册 A、B。下面是本课的实际配置思路，先阅读，不必立即执行：

```python
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"],
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    init_lora_weights=True,
    use_rslora=False,
    inference_mode=False,
)
model = get_peft_model(base, config, autocast_adapter_dtype=True)
```

本课明确固定为普通 LoRA 的 \(\alpha/r\)；`use_rslora=True` 会改变缩放规则，暂不混用。`bias="none"` 不训练原偏置，`modules_to_save=None` 不额外训练输出头。

这里选择 Q/V 是为了控制范围和成本，**不是所有任务的最佳配置结论**。

来源：[PEFT LoraConfig](https://huggingface.co/docs/peft/v0.18.0/package_reference/lora)。

## 2. 不猜层名：先列出真实模块，再注入

先用 `named_modules()` 找到普通 `Linear` 的完整路径、输入与输出维度。

本课采用**点分隔的末段匹配**：

```text
model.layers.0.self_attn.q_proj  → 命中 q_proj
notq_proj                      → 不命中
model.q_proj_backup            → 不命中
```

这不是随手写一个 `"q_proj" in name`。列表、正则字符串、`all-linear` 的含义也不同。真实 PEFT 支持多种匹配方式；本课使用已核对的目标路径集合检查结果，而不只相信“好像匹配到了”。

如果再次运行注入单元，原基座可能已经改过了。`get_peft_model()` **会就地修改传入模型**，不是深拷贝。应重建干净基座，不要重复叠加。

来源：[PEFT 注入行为](https://huggingface.co/docs/peft/guides/peft_model_config)。

### 2.1 模块规划、逐层审计与基座指纹

下一格实现这些可复用检查。哈希按块读取参数字节，避免为了验证“没改基座”而在 GPU 再复制整套参数。哈希会花时间，不把它计入推理性能。

In [3]:
def matches_target(name: str, target: str) -> bool:
    # 本课明确采用点分隔的末段匹配，不接受包含某个字符串就算命中。
    return name == target or name.endswith("." + target)


def lora_count(in_features: int, out_features: int, rank: int) -> int:
    for value, name in [(in_features, "in_features"), (out_features, "out_features"), (rank, "rank")]:
        positive_int(value, name)
    return rank * (in_features + out_features)


def plan_from_config(config: dict, rank: int = 8, targets=TARGETS) -> list[dict]:
    positive_int(rank, "rank")
    require(isinstance(targets, (tuple, list)) and 0 < len(targets) == len(set(targets)), "目标不能为空或重复")
    require(set(targets) <= {"q_proj", "k_proj", "v_proj", "o_proj"}, "本课只规划注意力投影")
    d = positive_int(config["hidden_size"], "hidden_size")
    hq = positive_int(config["num_attention_heads"], "num_attention_heads")
    hkv = positive_int(config["num_key_value_heads"], "num_key_value_heads")
    layers = positive_int(config["num_hidden_layers"], "num_hidden_layers")
    require(hq % hkv == 0, "Query头数必须能被KV头数整除")
    head_dim = config.get("head_dim")
    if head_dim is None:
        require(d % hq == 0, "没有显式 head_dim 且不能整除")
        head_dim = d // hq
    positive_int(head_dim, "head_dim")
    shapes = {"q_proj": (d, hq * head_dim), "k_proj": (d, hkv * head_dim),
              "v_proj": (d, hkv * head_dim), "o_proj": (hq * head_dim, d)}
    return [{"name": f"model.layers.{i}.self_attn.{target}", "target": target,
             "in_features": shapes[target][0], "out_features": shapes[target][1],
             "lora_parameters": lora_count(*shapes[target], rank)}
            for i in range(layers) for target in targets]


def discover_targets(base: nn.Module, rank: int, targets=TARGETS) -> list[dict]:
    require(not any(hasattr(m, "lora_A") for m in base.modules()), "已有 adapter：请重建干净基座，禁止重复注入")
    rows = []
    for name, layer in base.named_modules():
        matching = [t for t in targets if matches_target(name, t)]
        if not matching:
            continue
        require(type(layer) is nn.Linear, f"{name} 不是本课支持的普通 Linear；不接量化层")
        require(len(matching) == 1, f"目标匹配歧义：{name}")
        rows.append({"name": name, "target": matching[0], "in_features": layer.in_features,
                     "out_features": layer.out_features,
                     "lora_parameters": lora_count(layer.in_features, layer.out_features, rank)})
    require(all(any(r["target"] == t for r in rows) for t in targets), "某个目标没有命中")
    return rows


def verify_inventory(actual: list[dict], expected: list[dict]) -> None:
    require(len({r["name"] for r in actual}) == len(actual), "重复目标路径")
    a, b = {r["name"]: r for r in actual}, {r["name"]: r for r in expected}
    require(a.keys() == b.keys(), "实际层路径与配置规划不符，不能只看命中数量")
    for name in a:
        for key in ("in_features", "out_features", "lora_parameters"):
            require(a[name][key] == b[name][key], f"{name} 的 {key} 不符")


def hash_named_parameters(named_parameters: Iterable, chunk_elements: int = 1 << 20) -> str:
    """逐块流式哈希；不在GPU复制整套基座，也不一次转成巨大Python列表。

    逐块拷到CPU读取字节会同步/耗时，本检查不计入任何速度基准。
    名字来自注入前保留的原Parameter引用，避免PEFT改名影响比较。
    """
    h = hashlib.sha256()
    for name, parameter in named_parameters:
        h.update(f"{name}|{list(parameter.shape)}|{parameter.dtype}|".encode())
        flat = parameter.detach().reshape(-1)
        for start in range(0, flat.numel(), chunk_elements):
            raw = flat[start:start + chunk_elements].to("cpu").contiguous().view(torch.uint8)
            h.update(raw.numpy().tobytes())
    return h.hexdigest()


def lora_spec(s: Settings) -> dict:
    return dict(r=s.rank, lora_alpha=s.alpha, target_modules=list(TARGETS),
                lora_dropout=0.0, bias="none", init_lora_weights=True,
                use_rslora=False, inference_mode=False, modules_to_save=None)


def audit_adapter(model, inventory: list[dict], s: Settings, *, initial=False,
                  trainable=True) -> dict:
    """根据结构对象收集A/B，再与requires_grad参数集合做精确比较。
    不以名称中碰巧含有'lora'作为唯一依据。
    """
    base = model.get_base_model()
    expected_ids, rows = set(), []
    found = {name for name, m in base.named_modules() if hasattr(m, "lora_A") and hasattr(m, "lora_B")}
    require(found == {r["name"] for r in inventory}, "注入路径集合有漏注入或多注入")
    for row in inventory:
        layer = base.get_submodule(row["name"])
        require(set(layer.lora_A.keys()) == {"default"} and set(layer.lora_B.keys()) == {"default"},
                "本课只接受单个 default adapter")
        a = layer.lora_A["default"].weight
        b = layer.lora_B["default"].weight
        require(tuple(a.shape) == (s.rank, row["in_features"]), "A形状错误")
        require(tuple(b.shape) == (row["out_features"], s.rank), "B形状错误")
        require(math.isclose(float(layer.scaling["default"]), s.alpha / s.rank), "scaling不是alpha/r")
        require(a.dtype == torch.float32 and b.dtype == torch.float32, "本课要求可训练adapter保留FP32")
        require(bool(torch.isfinite(a).all() and torch.isfinite(b).all()), "adapter包含非有限值")
        require(a.requires_grad == trainable and b.requires_grad == trainable, "adapter可训练状态错误")
        for p in (a, b):
            expected_ids.add(id(p))
        if initial:
            require(bool(torch.count_nonzero(a) > 0), "A不应全零")
            require(bool(torch.count_nonzero(b) == 0), "默认B初始化应为零")
        rows.append(dict(row, a_shape=list(a.shape), b_shape=list(b.shape),
                         a_dtype=str(a.dtype), b_dtype=str(b.dtype), scaling=float(layer.scaling["default"])))
    live_ids = {id(p) for p in model.parameters() if p.requires_grad}
    require(live_ids == (expected_ids if trainable else set()), "存在意外可训练参数、漏冻基座或漏训adapter")
    total = sum(p.numel() for p in model.parameters())
    active = sum(p.numel() for p in model.parameters() if p.requires_grad)
    expected = sum(r["lora_parameters"] for r in inventory)
    require(not trainable or active == expected, "可训练参数总数不符")
    return {"target_modules": len(rows), "adapter_tensor_count": 2 * len(rows),
            "expected_adapter_parameters": expected, "trainable_parameters": active,
            "total_parameters": total, "trainable_percent": 100 * active / total,
            "rows": rows}


def assert_optimizer_matches(model: nn.Module, optimizer) -> None:
    expected = {id(p) for p in model.parameters() if p.requires_grad}
    params = [p for group in optimizer.param_groups for p in group["params"]]
    require(len(params) == len({id(p) for p in params}), "优化器参数重复")
    require(bool(expected) and {id(p) for p in params} == expected, "优化器应且仅应包含当前可训练参数；请在注入后创建")

### 2.2 Qwen3-0.6B 的参数预算：不要忘记显式 `head_dim`

固定仓库配置：隐藏维度 1024、Query 头数 16、KV 头数 8、**每头 128 维**，28 层。

所以：

- Q 投影：`1024 → 2048`，A/B 参数数为 \(r(1024+2048)\)。
- V 投影：`1024 → 1024`，A/B 参数数为 \(r(1024+1024)\)。

取 r=8：
\[
28\times 8\times[(1024+2048)+(1024+1024)]=1,146,880.
\]

这是**从固定配置推导的预算**，不是本次加载权重的测量值。随后还需与实际模块维度、`requires_grad` 参数集合比较。

来源：[固定配置](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/config.json)。

In [4]:
plan = plan_from_config(CONFIG_EXAMPLE, rank=8)
print("预期注入模块：", len(plan))
print("预期 A/B 张量：", 2 * len(plan))
print("预期 adapter 参数：", sum(r["lora_parameters"] for r in plan))
print(json.dumps(plan[:2], ensure_ascii=False, indent=2))
print("r=4 时参数：", sum(r["lora_parameters"] for r in plan_from_config(CONFIG_EXAMPLE, 4)))

预期注入模块： 56
预期 A/B 张量： 112
预期 adapter 参数： 1146880
[
  {
    "name": "model.layers.0.self_attn.q_proj",
    "target": "q_proj",
    "in_features": 1024,
    "out_features": 2048,
    "lora_parameters": 24576
  },
  {
    "name": "model.layers.0.self_attn.v_proj",
    "target": "v_proj",
    "in_features": 1024,
    "out_features": 1024,
    "lora_parameters": 16384
  }
]
r=4 时参数： 573440


In [5]:
for name in ["model.layers.0.self_attn.q_proj", "notq_proj", "model.q_proj_backup"]:
    print(name, "→", matches_target(name, "q_proj"))

bad = copy.deepcopy(plan)
bad[0]["out_features"] = 1024  # 错误地按D/H推导后，Q宽度会算错。
try:
    verify_inventory(bad, plan)
except ValueError as error:
    print("预期的审计拒绝：", error)

model.layers.0.self_attn.q_proj → True
notq_proj → False
model.q_proj_backup → False
预期的审计拒绝： model.layers.0.self_attn.q_proj 的 out_features 不符


## 3. 初始输出相同，是一次接线检查，不是训练结果

本课使用默认 A 随机、B 全零，因此初始增量为零。检查时：

1. 注入前，用固定输入取得基座 logits。
2. 注入后，保持相同的 dtype、设备、前向方式与 `eval()` 模式。
3. 比较**整个 logits 张量**，而不是只比较生成的一个词。
4. 审计 A/B 形状、缩放和初值。

必须在注入前保存基座输出，不能把两个指向同一个已修改模型的变量当作两个独立对照。

`autocast_adapter_dtype=True` 会在适用情况下把低精度 adapter 参数提升到 FP32；本课随后核对实际 dtype，不假定所有参数与基座相同。不要在注入后对整个模型调用 `.half()`，把刚保留的 FP32 adapter 一起降精度。

来源：[PEFT 模型接口与 adapter dtype](https://huggingface.co/docs/peft/v0.18.0/en/package_reference/peft_model)。

## 4. 从手写模型切到 Hugging Face：标签契约变了

以前我们自己构造 `x = ids[:-1]`、`y = ids[1:]`，再调用手写 loss。

本课的 Qwen3 `model(..., labels=labels)` **内部会完成一次 next-token 移位**。传入的标签应该与输入同位，只把不希望计入目标的位置替换为 `-100`：

```text
位置：       0   1   2   3   4   5
input_ids：  1   3   4   5   6   2
labels：  -100 -100 -100  5   6   2
```

位置 2 的 logits 预测位置 3 的 token 5；最后一个输入位置不再额外预测自己。

**不能再提前移位一次。** 也不要把 `-100` 放到输入编号中，它不是 Embedding 可用的 token。

来源：[所用版本的 Qwen3 源码](https://raw.githubusercontent.com/huggingface/transformers/v4.57.3/src/transformers/models/qwen3/modeling_qwen3.py)、[ForCausalLMLoss](https://raw.githubusercontent.com/huggingface/transformers/v4.57.3/src/transformers/loss/loss_utils.py)。

### 4.1 标签与模板工具

本课先支持**一个 user 提问＋一个末尾 assistant 回答**，不冒充任意多轮数据整理器。

先分别用同一个模板编码“提问＋assistant 前缀”和“完整对话”，逐 token 验证前缀完全一致。对不上就报错，不拿字符串长度猜 mask 边界。

真实 Qwen 样本的空 thinking 前缀属于提示，不参与监督。回答与 assistant 结束 token 参与监督；只去掉结束标记后的模板空白，绝不静默截断答案。

来源：[固定 Qwen3 模板](https://huggingface.co/Qwen/Qwen3-0.6B/raw/c1899de289a04d12100db370d81485cdf75e47ca/tokenizer_config.json)。

In [6]:
def valid_ids(ids, name="ids") -> list[int]:
    require(isinstance(ids, list) and all(type(t) is int and t >= 0 for t in ids),
            f"{name}必须是非负整数列表，不接受bool")
    return ids


def row_from_verified_prefix(prefix_ids: list[int], full_ids: list[int], max_length: int) -> dict:
    """prefix与full必须已用同一模板/tokenizer编码。只支持一条末尾assistant回答。"""
    valid_ids(prefix_ids, "prefix")
    valid_ids(full_ids, "full")
    positive_int(max_length, "max_length")
    n = len(prefix_ids)
    require(n > 0 and n < len(full_ids), "需要非空前缀和至少一个监督目标")
    require(full_ids[:n] == prefix_ids, "token前缀不一致；不能靠字符串长度猜mask边界")
    require(len(full_ids) <= max_length, "样本超长；本课拒绝静默截断答案")
    return {"input_ids": full_ids.copy(), "labels": [-100] * n + full_ids[n:], "prefix_length": n}


def collate_rows(rows: list[dict], pad_id: int) -> dict[str, torch.Tensor]:
    require(bool(rows), "空batch")
    require(type(pad_id) is int and pad_id >= 0, "pad_id不合法")
    length = max(len(r["input_ids"]) for r in rows)
    ids, labels, masks = [], [], []
    for row in rows:
        tokens, target = row["input_ids"], row["labels"]
        valid_ids(tokens)
        require(len(tokens) == len(target) and len(tokens) >= 2, "标签长度错误")
        require(all(t == -100 or (type(t) is int and t == tokens[i]) for i, t in enumerate(target)),
                "标签必须与输入同位，不能提前移位")
        n = len(tokens)
        ids.append(tokens + [pad_id] * (length - n))
        labels.append(target + [-100] * (length - n))
        masks.append([1] * n + [0] * (length - n))
    batch = {"input_ids": torch.tensor(ids, dtype=torch.long),
             "attention_mask": torch.tensor(masks, dtype=torch.long),
             "labels": torch.tensor(labels, dtype=torch.long)}
    require(bool((batch["labels"][:, 1:] != -100).any()), "没有有效的next-token目标")
    return batch


def manual_causal_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    require(logits.ndim == 3 and labels.ndim == 2 and logits.shape[:2] == labels.shape, "logits/labels形状不匹配")
    require(labels.dtype == torch.long and logits.shape[1] >= 2, "标签dtype或长度错误")
    targets = labels[:, 1:]
    require(bool((targets != -100).any()), "全部标签被忽略，loss没有意义")
    return F.cross_entropy(logits[:, :-1, :].float().reshape(-1, logits.shape[-1]),
                           targets.reshape(-1), ignore_index=-100)


def supervised_positions(batch: dict) -> list[dict]:
    result = []
    for row in range(batch["labels"].shape[0]):
        for j in range(1, batch["labels"].shape[1]):
            if batch["labels"][row, j].item() != -100:
                result.append({"row": row, "logits_position": j - 1,
                               "target_position": j, "target_id": batch["labels"][row, j].item()})
    return result


def encode_one_turn(tokenizer, prompt: str, answer: str, max_length: int) -> tuple[dict, dict]:
    """真实Qwen模板入口。验证token前缀，不依赖可能缺少generation标记的assistant_masks。"""
    require(isinstance(prompt, str) and prompt.strip(), "prompt不能为空")
    require(isinstance(answer, str) and answer.strip(), "answer不能为空")
    # 本课不用模板控制token作为内容，避免结束边界歧义；这不是通用prompt-injection防护。
    for text in (prompt, answer):
        require(not any(x in text for x in ("<|im_start|>", "<|im_end|>", "<think>", "</think>")),
                "本课样本禁止嵌入模板控制标记")
    messages = [{"role": "user", "content": prompt}]
    prefix = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           enable_thinking=False)
    full = tokenizer.apply_chat_template(messages + [{"role": "assistant", "content": answer}],
                                         tokenize=True, add_generation_prompt=False, enable_thinking=False)
    prefix, full = list(prefix), list(full)
    require(full[:len(prefix)] == prefix, "此模板/分词边界不是稳定前缀，本课拒绝猜测")
    tail = full[len(prefix):]
    eos = tokenizer.eos_token_id
    require(type(eos) is int and eos in tail, "assistant结束token不存在")
    eos_index = len(prefix) + tail.index(eos)
    require(eos_index > len(prefix), "不能只训练一个结束标记")
    trailing = full[eos_index + 1:]
    require(tokenizer.decode(trailing, skip_special_tokens=False).strip() == "",
            "结束标记后有非空内容，不能按单轮样本裁掉")
    # 仅去掉固定模板的尾随空白，不截断答案；EOS仍参与监督。
    kept = full[:eos_index + 1]
    row = row_from_verified_prefix(prefix, kept, max_length)
    details = {"prompt": prompt, "answer": answer, "prefix_length": len(prefix),
               "input_length": len(kept), "supervised_ids": kept[len(prefix):],
               "supervised_text": tokenizer.decode(kept[len(prefix):], skip_special_tokens=False),
               "removed_trailing_whitespace_ids": trailing, "prefix_ids": prefix}
    return row, details

In [7]:
row1 = row_from_verified_prefix([1, 3, 4], [1, 3, 4, 5, 6, 2], 12)
row2 = row_from_verified_prefix([1, 7], [1, 7, 8, 2], 12)
batch_demo = collate_rows([row1, row2], pad_id=0)
for name, value in batch_demo.items():
    print(name, "\n", value)
print("有效的预测-目标配对：")
for record in supervised_positions(batch_demo):
    print(record)

input_ids 
 tensor([[1, 3, 4, 5, 6, 2],
        [1, 7, 8, 2, 0, 0]])
attention_mask 
 tensor([[1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0]])
labels 
 tensor([[-100, -100, -100,    5,    6,    2],
        [-100, -100,    8,    2, -100, -100]])
有效的预测-目标配对：
{'row': 0, 'logits_position': 2, 'target_position': 3, 'target_id': 5}
{'row': 0, 'logits_position': 3, 'target_position': 4, 'target_id': 6}
{'row': 0, 'logits_position': 4, 'target_position': 5, 'target_id': 2}
{'row': 1, 'logits_position': 1, 'target_position': 2, 'target_id': 8}
{'row': 1, 'logits_position': 2, 'target_position': 3, 'target_id': 2}


### 4.2 忽略提示的 loss，不等于忽略提示作为上下文

`labels=-100` 决定哪些**目标位置**计入损失；`attention_mask` 决定哪些输入位置是有效文本。两者职责不同。

本课训练时使用**右补齐**，按原始长度标记 padding。即使 PAD 与 EOS 碰巧使用同一个编号，也不能把所有等于该 ID 的标签全部删掉，否则会失去学习结束的目标。

下面直接观察有监督的 logits 位置以及一个“双重移位”错误。

In [8]:
probe = torch.zeros(1, 4, 8, requires_grad=True)
probe_labels = torch.tensor([[-100, -100, 4, 2]])
probe_loss = manual_causal_loss(probe, probe_labels)
probe_loss.backward()
print("均匀分布的loss：", probe_loss.item(), "；log(8)：", math.log(8))
print("每个logits位置的梯度绝对值总和：", probe.grad.abs().sum(-1))

perfectish = torch.zeros(1, 4, 8)
perfectish[0, 1, 4] = 8.0
perfectish[0, 2, 2] = 8.0
wrongly_shifted = torch.tensor([[-100, 4, 2, -100]])
print("正确同位labels：", manual_causal_loss(perfectish, probe_labels).item())
print("提前移位的错误labels：", manual_causal_loss(perfectish, wrongly_shifted).item())

均匀分布的loss： 2.079441547393799 ；log(8)： 2.0794415416798357
每个logits位置的梯度绝对值总和： tensor([[0.0000, 0.8750, 0.8750, 0.0000]])
正确同位labels： 0.0023454318288713694
提前移位的错误labels： 5.0408935546875


In [9]:
same_pad_eos = collate_rows([
    row_from_verified_prefix([1, 3], [1, 3, 4, 2], 8),
    row_from_verified_prefix([1], [1, 2], 8),
], pad_id=2)
print("PAD=EOS的输入：\n", same_pad_eos["input_ids"])
print("按位置遮掉padding后的标签：\n", same_pad_eos["labels"])
print("真实EOS仍然受监督，不因编号相同而被删除。")

PAD=EOS的输入：
 tensor([[1, 3, 4, 2],
        [1, 2, 2, 2]])
按位置遮掉padding后的标签：
 tensor([[-100, -100,    4,    2],
        [-100,    2, -100, -100]])
真实EOS仍然受监督，不因编号相同而被删除。


## 5. 每一步都核对：loss → 梯度 → 参数更新

我们检查三个不同现象：

| 观察 | 可能含义 |
|---|---|
| `.grad is None` | 参数没有参与这次反传，或梯度已清除；需结合检查时机判断 |
| 梯度是全零张量 | 已建立反传，但当前导数为零；B=0时A首步为零就合理 |
| 梯度有限且非零 | 收到了训练信号，但仍需验证优化器真的更新了这个参数 |

优化器应当在 adapter 注入以后创建，并且仅持有当前 A/B 参数。只运行 `print_trainable_parameters()` 不能证明优化器包含了它们。

冻结基座不等于把整个前向放进 `no_grad()`。后者会切断 adapter 所需反传。

本课更新顺序：清零 → 前向与内置loss → 核对手算loss → 反传 → 去缩放 → 梯度审计 → 裁剪 → step/update。CUDA FP16 时启用 GradScaler，CPU FP32 不启用。

### 5.1 公用诊断程序

此格定义真实 PEFT 路径与离线替身都会调用的**检查逻辑**，不声称两种运行环境等价。

学习时重点阅读 `one_update()` 和 `diagnose()`。完整实现也检查基座哈希、optimizer 参数身份、A/B 的梯度，以及禁用 adapter 后是否回到初始基座输出。

In [10]:
def snapshot_trainable(model) -> dict:
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def parameter_changes(model, previous: dict) -> dict:
    current = dict(model.named_parameters())
    changes = {name: float((current[name].detach().cpu() - old).abs().max())
               for name, old in previous.items()}
    return {"changed_tensors": sum(v > 0 for v in changes.values()), "max_abs_change": max(changes.values(), default=0.0),
            "by_name": changes}


def gradient_report(model, inventory) -> dict:
    base = model.get_base_model()
    groups = {"A": [], "B": []}
    missing, nonfinite = [], []
    adapter_ids = set()
    for row in inventory:
        layer = base.get_submodule(row["name"])
        for label, parameter in [("A", layer.lora_A["default"].weight), ("B", layer.lora_B["default"].weight)]:
            adapter_ids.add(id(parameter))
            name = row["name"] + ".lora_" + label
            if parameter.grad is None:
                missing.append(name)
            elif not bool(torch.isfinite(parameter.grad).all()):
                nonfinite.append(name)
            else:
                groups[label].append(float(parameter.grad.detach().float().norm()))
    frozen_with_grad = [n for n, p in model.named_parameters() if id(p) not in adapter_ids and p.grad is not None]
    return {"A_grad_norm": math.sqrt(sum(v*v for v in groups["A"])),
            "B_grad_norm": math.sqrt(sum(v*v for v in groups["B"])),
            "A_nonzero_tensors": sum(v > 0 for v in groups["A"]),
            "B_nonzero_tensors": sum(v > 0 for v in groups["B"]),
            "missing": missing, "nonfinite": nonfinite, "frozen_with_grad": frozen_with_grad}


def one_update(model, batch, inventory, optimizer, scaler, device, dtype) -> dict:
    assert_optimizer_matches(model, optimizer)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    with amp_context(device, dtype):
        output = model(**batch, use_cache=False, return_dict=True)
        loss = output.loss
    require(loss.ndim == 0 and bool(torch.isfinite(loss)), "loss不是有限标量")
    reference_loss = manual_causal_loss(output.logits.detach(), batch["labels"])
    torch.testing.assert_close(loss.detach().float(), reference_loss, atol=3e-5, rtol=3e-5)
    loss_value = float(loss.detach())
    valid_count = int((batch["labels"][:, 1:] != -100).sum())
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    grads = gradient_report(model, inventory)
    require(not grads["missing"], "某些adapter没有梯度，请检查激活路径")
    require(not grads["nonfinite"] and not grads["frozen_with_grad"], "非有限梯度或基座意外得到梯度")
    norm = float(nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0,
                                        error_if_nonfinite=True))
    old_scale = float(scaler.get_scale())
    scaler.step(optimizer)
    scaler.update()
    new_scale = float(scaler.get_scale())
    require(new_scale >= old_scale, "冒烟更新被GradScaler跳过；请检查数值状态，不计作成功")
    optimizer.zero_grad(set_to_none=True)
    return {"loss": loss_value, "effective_target_tokens": valid_count, "gradients": grads,
            "pre_clip_grad_norm": norm, "scale_before": old_scale, "scale_after": new_scale}


def eval_logits(model, batch, device, dtype) -> torch.Tensor:
    previous = model.training
    model.eval()
    try:
        with torch.no_grad(), amp_context(device, dtype):
            output = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                           use_cache=False, return_dict=True)
            require(bool(torch.isfinite(output.logits).all()), "非有限logits")
            return output.logits.detach().float().cpu()
    finally:
        model.train(previous)


def max_difference(a: torch.Tensor, b: torch.Tensor) -> float:
    require(a.shape == b.shape, "比较张量形状不一致")
    return float((a.float() - b.float()).abs().max())


def close_outputs(a, b, dtype):
    atol = 2e-5 if dtype == torch.float32 else (5e-3 if dtype == torch.float16 else 3e-2)
    torch.testing.assert_close(a, b, atol=atol, rtol=atol)


def diagnose(model, batch, inventory, s, original_refs, original_hash, baseline, device, dtype):
    initial_audit = audit_adapter(model, inventory, s, initial=True)
    injected = eval_logits(model, batch, device, dtype)
    close_outputs(injected, baseline, dtype)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=s.lr, weight_decay=0.0)
    assert_optimizer_matches(model, optimizer)
    scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda" and dtype == torch.float16))
    before = snapshot_trainable(model)
    history = []
    for step in range(s.steps):
        record = one_update(model, batch, inventory, optimizer, scaler, device, dtype)
        record["step"] = step + 1
        history.append(record)
    changes = parameter_changes(model, before)
    if s.steps:
        require(changes["changed_tensors"] > 0, "没有任何adapter参数变化")
        require(history[0]["gradients"]["A_grad_norm"] == 0.0, "默认B=0时A首步梯度应为0")
        require(history[0]["gradients"]["B_grad_norm"] > 0, "B首步梯度没有激活")
    final = eval_logits(model, batch, device, dtype)
    with model.disable_adapter():
        disabled = eval_logits(model, batch, device, dtype)
    close_outputs(disabled, baseline, dtype)
    after_hash = hash_named_parameters(original_refs)
    require(after_hash == original_hash, "基座参数发生变化！")
    require(all(not p.requires_grad and p.grad is None for _, p in original_refs), "基座没有完全冻结/清梯度")
    final_loss = float(manual_causal_loss(final, batch["labels"].cpu()))
    return {"initial_audit": initial_audit,
            "initial_logits_max_diff": max_difference(injected, baseline),
            "disabled_logits_max_diff": max_difference(disabled, baseline),
            "adapted_logits_max_diff": max_difference(final, baseline),
            "base_sha256_before": original_hash, "base_sha256_after": after_hash,
            "base_unchanged": True, "history": history, "parameter_changes": changes,
            "final_probe_loss": final_loss,
            "meaning": "注入与梯度冒烟检查；少量同样本更新不是微调质量/泛化评测"}, final

### 5.2 离线接口替身：仅验证我们的检查逻辑

以下小模型使用普通 PyTorch 实现一个可检查的计算图。它的名字包含 `Contract`，报告标记为 `NOT_PEFT_NOT_QWEN`。

它既不是预训练模型，也不是“本机成功加载 PEFT”的替代证据。作用是让你在尚未下载依赖前，验证标签、参数集合、梯度和模式恢复等逻辑。

In [11]:
class ContractLoRALinear(nn.Module):
    """只供本课离线断言测试的PyTorch替身；不是PEFT层，不能保存PEFT adapter。"""
    def __init__(self, base: nn.Linear, rank: int, alpha: int):
        super().__init__()
        self.base_layer = base
        self.base_layer.requires_grad_(False)
        self.lora_A = nn.ModuleDict({"default": nn.Linear(base.in_features, rank, bias=False)})
        self.lora_B = nn.ModuleDict({"default": nn.Linear(rank, base.out_features, bias=False)})
        nn.init.zeros_(self.lora_B["default"].weight)
        self.scaling = {"default": alpha / rank}
        self.enabled = True
    def forward(self, x):
        out = self.base_layer(x)
        if self.enabled:
            out = out + self.scaling["default"] * self.lora_B["default"](self.lora_A["default"](x))
        return out


class ContractAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = nn.Linear(8, 8, bias=False)
        self.k_proj = nn.Linear(8, 4, bias=False)
        self.v_proj = nn.Linear(8, 4, bias=False)
        self.o_proj = nn.Linear(8, 8, bias=False)
    def forward(self, x, mask):
        b, t, _ = x.shape
        q = self.q_proj(x).reshape(b, t, 2, 4).transpose(1, 2)
        k = self.k_proj(x).reshape(b, t, 1, 4).transpose(1, 2)
        v = self.v_proj(x).reshape(b, t, 1, 4).transpose(1, 2)
        allowed = torch.ones(t, t, dtype=torch.bool, device=x.device).tril()[None, None] & mask[:, None, None, :].bool()
        scores = (q @ k.transpose(-1, -2)) / 2.0
        weights = scores.masked_fill(~allowed, -torch.inf).softmax(-1)
        out = (weights @ v).transpose(1, 2).reshape(b, t, 8)
        return self.o_proj(out)


class ContractLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.config = SimpleNamespace(hidden_size=8, num_attention_heads=2, num_key_value_heads=1,
                                      head_dim=4, num_hidden_layers=2, vocab_size=32, use_cache=False)
        self.model = nn.Module()
        self.model.embed_tokens = nn.Embedding(32, 8)
        self.model.layers = nn.ModuleList([nn.Module() for _ in range(2)])
        for layer in self.model.layers:
            layer.self_attn = ContractAttention()
        self.lm_head = nn.Linear(8, 32, bias=False)
    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        x = self.model.embed_tokens(input_ids)
        for layer in self.model.layers:
            x = x + layer.self_attn(F.layer_norm(x, (8,)), attention_mask)
        logits = self.lm_head(x)
        return SimpleNamespace(logits=logits, loss=None if labels is None else manual_causal_loss(logits, labels))


class ContractPeftFixture(nn.Module):
    """只为测试审计/优化器/模式恢复协议；不冒充get_peft_model已被执行。"""
    def __init__(self, base, s):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        for row in discover_targets(base, s.rank):
            parent_path, name = row["name"].rsplit(".", 1)
            parent = base.get_submodule(parent_path)
            setattr(parent, name, ContractLoRALinear(getattr(parent, name), s.rank, s.alpha))
    def get_base_model(self):
        return self.base
    def forward(self, **kwargs):
        return self.base(**kwargs)
    @contextlib.contextmanager
    def disable_adapter(self):
        layers = [m for m in self.modules() if isinstance(m, ContractLoRALinear)]
        old = [m.enabled for m in layers]
        for m in layers:
            m.enabled = False
        try:
            yield
        finally:
            for m, enabled in zip(layers, old):
                m.enabled = enabled


def fixture_batch():
    # 人工编号，不是Qwen token；两条样本长度不同。
    rows = [row_from_verified_prefix([1, 3, 4], [1, 3, 4, 5, 6, 2], 12),
            row_from_verified_prefix([1, 7], [1, 7, 8, 2], 12)]
    return collate_rows(rows, pad_id=0)


def make_contract_fixture(s=None):
    s = s or Settings(rank=2, alpha=4)
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(SEED)
        base = ContractLM()
        inventory = discover_targets(base, s.rank)
        refs = list(base.named_parameters())
        model = ContractPeftFixture(base, s)
    return model, inventory, refs


def offline_demo() -> dict:
    s = Settings(rank=2, alpha=4, steps=2, lr=0.01)
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(SEED)
        base = ContractLM()
        batch = fixture_batch()
        device, dtype = torch.device("cpu"), torch.float32
        refs = list(base.named_parameters())
        before_hash = hash_named_parameters(refs)
        baseline = eval_logits(base, batch, device, dtype)
        inventory = discover_targets(base, s.rank)
        model = ContractPeftFixture(base, s)
        report, _ = diagnose(model, batch, inventory, s, refs, before_hash, baseline, device, dtype)
    plan = plan_from_config(CONFIG_EXAMPLE, 8)
    return {"kind": "pytorch_contract_fixture_NOT_PEFT_NOT_QWEN",
            "config_example_not_weight_measurement": {"target_count": len(plan),
            "adapter_parameters": sum(r["lora_parameters"] for r in plan), "first_layer": plan[:2]},
            "labels_demo": {k: v.tolist() for k, v in batch.items()},
            "supervised_positions": supervised_positions(batch), "diagnostics": report}

In [12]:
offline = offline_demo()
print("实验类型：", offline["kind"])
for item in offline["diagnostics"]["history"]:
    gradients = item["gradients"]
    print(f"step={item['step']} loss={item['loss']:.6f} "
          f"A_grad={gradients['A_grad_norm']:.6f} B_grad={gradients['B_grad_norm']:.6f}")
print("注入前后初始logits最大差异：", offline["diagnostics"]["initial_logits_max_diff"])
print("基座字节哈希保持相同：", offline["diagnostics"]["base_unchanged"])
print("禁用adapter后logits最大差异：", offline["diagnostics"]["disabled_logits_max_diff"])
print("仅是两个更新的机制实验，不是质量提升。")

实验类型： pytorch_contract_fixture_NOT_PEFT_NOT_QWEN
step=1 loss=3.298691 A_grad=0.000000 B_grad=0.207934
step=2 loss=3.288947 A_grad=0.014394 B_grad=0.206719
注入前后初始logits最大差异： 0.0
基座字节哈希保持相同： True
禁用adapter后logits最大差异： 0.0
仅是两个更新的机制实验，不是质量提升。


### 5.3 首步 A=0 梯度与第二步非零梯度

\[
\frac{\partial L}{\partial A}=\frac{\alpha}{r}B^T G
\]

默认 B=0，第一步 A 梯度为零；B 接收到梯度并更新后，下一步 A 才可能收到非零梯度。

本课关闭 weight decay，以免“无梯度信号也因衰减发生参数变化”干扰这个小实验。注意**不能要求每个真实任务的每张 A 在任意时刻都非零**；实际报告逐层记录。

看 loss 下降还不够。先确认只更新了预期参数，再考虑任务指标。

In [13]:
fixture, inventory, references = make_contract_fixture()
right_optimizer = torch.optim.AdamW(
    [p for p in fixture.parameters() if p.requires_grad], lr=1e-3, weight_decay=0.0
)
assert_optimizer_matches(fixture, right_optimizer)
print("当前adapter参数与优化器完全匹配。")
wrong_optimizer = torch.optim.AdamW([p for _, p in references], lr=1e-3)
try:
    assert_optimizer_matches(fixture, wrong_optimizer)
except ValueError as error:
    print("旧optimizer被拒绝：", error)
del fixture, right_optimizer, wrong_optimizer, references

当前adapter参数与优化器完全匹配。
旧optimizer被拒绝： 优化器应且仅应包含当前可训练参数；请在注入后创建


## 6. 真实库集成分三步打开，不直接下载后就盲目训练

本课按照下列官方版本接口编写：

```bash
python -m pip install "transformers==4.57.3" "peft==0.18.0" "accelerate==1.12.0" safetensors
```

这是**用于课程隔离环境的固定组合建议，不是已在制作环境验证的兼容矩阵，也不宣称是最新版本**。当前已尝试安装，但环境无法解析 PyPI 域名；因此下方真实库路径未实测。保留你已经能调用 CUDA 的 PyTorch，不要为本课盲目重装驱动。

建议先通过 `--local-peft`，再 `--inspect`，最后 `--run`。三个阶段都保存真实失败信息，失败不会伪装成成功。

已有新版本的重要项目，使用独立环境而非原地降级。

### 6.1 真实 PEFT 与真实模型入口

此格只定义函数，不执行下载。关键部分是 `inject_with_peft()`、`run_library_experiment()` 与 `save_adapter_checked()`。

真实路径固定：单设备普通浮点基座、Q/V LoRA、无量化、无 `device_map="auto"`、不卸载、短序列、训练时关闭 KV cache。可选非重入激活检查点；不是开启后就保证所有显存容量都能运行。

保存后先释放原模型，再加载同一基座并恢复 adapter，避免同时保留两份大基座。

In [14]:
def small_qwen_config(hf):
    # 显式head_dim=16，不等于48/4；同时验证不能乱用D/H。
    return hf.Qwen3Config(vocab_size=64, hidden_size=48, intermediate_size=128,
                          num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=2,
                          head_dim=16, max_position_embeddings=64, bos_token_id=1,
                          eos_token_id=2, pad_token_id=0, tie_word_embeddings=True,
                          attention_dropout=0.0, use_cache=False)


def pretrained_inspection(hf, s: Settings):
    common = dict(revision=REVISION, local_files_only=s.local_files_only, trust_remote_code=False)
    config = hf.AutoConfig.from_pretrained(MODEL_ID, **common)
    require(config.model_type == "qwen3", "不是本课预期的Qwen3配置")
    tok = hf.AutoTokenizer.from_pretrained(MODEL_ID, **common)
    require(tok.pad_token_id is not None and tok.eos_token_id is not None, "缺少pad/eos约定")
    require(isinstance(tok.chat_template, str) and bool(tok.chat_template), "没有配套chat template")
    plan = plan_from_config(config.to_dict(), s.rank)
    row, detail = encode_one_turn(tok, "请只回复：你好。", "你好。", s.max_length)
    batch = collate_rows([row], tok.pad_token_id)
    require(int(batch["input_ids"].max()) < config.vocab_size, "token ID超过模型词表范围")
    info = {"kind": "real_config_tokenizer_only_NO_WEIGHTS", "model_id": MODEL_ID, "revision": REVISION,
            "config": config.to_dict(), "expected_targets": plan,
            "expected_adapter_parameters": sum(r["lora_parameters"] for r in plan),
            "template_sha256": hashlib.sha256(tok.chat_template.encode()).hexdigest(),
            "sample": detail, "batch": {k: v.tolist() for k, v in batch.items()},
            "alignment": supervised_positions(batch)}
    return config, tok, batch, info


def load_real_base(hf, s, device, dtype):
    base = hf.AutoModelForCausalLM.from_pretrained(
        MODEL_ID, revision=REVISION, local_files_only=s.local_files_only,
        trust_remote_code=False, dtype=dtype, attn_implementation="eager",
        low_cpu_mem_usage=True,
    )
    # 不用device_map=auto，不分片/卸载，不接第31课的量化对象。
    base.to(device)
    require({p.device.type for p in base.parameters()} == {device.type}, "存在意外设备/卸载")
    base.config.use_cache = False
    return base


def inject_with_peft(hf, base, s):
    # 显式拒绝二次注入；get_peft_model会改动传入模型，不是深拷贝。
    discover_targets(base, s.rank)
    config = hf.LoraConfig(task_type=hf.TaskType.CAUSAL_LM, **lora_spec(s))
    revision = REVISION if getattr(base.config, "_name_or_path", "") == MODEL_ID else None
    return hf.get_peft_model(base, config, autocast_adapter_dtype=True,
                             low_cpu_mem_usage=False, revision=revision)


def save_adapter_checked(model, path: Path) -> dict:
    model.save_pretrained(str(path), safe_serialization=True, save_embedding_layers=False)
    config_file, weights_file = path / "adapter_config.json", path / "adapter_model.safetensors"
    require(config_file.is_file() and weights_file.is_file(), "adapter文件没有完整写出")
    from safetensors.torch import load_file
    state = load_file(str(weights_file))
    require(bool(state), "空adapter文件")
    require(all(".lora_A." in n or ".lora_B." in n for n in state), "意外保存了基座/其他参数")
    require(all(bool(torch.isfinite(v).all()) for v in state.values()), "保存的adapter包含非有限值")
    return {"files": sorted(p.name for p in path.iterdir()), "adapter_tensors": len(state),
            "tensor_parameters": sum(t.numel() for t in state.values()),
            "file_bytes": weights_file.stat().st_size(), "keys": sorted(state)}


def run_library_experiment(kind: str, s: Settings) -> Path:
    require(kind in {"local_peft", "real"}, "未知实验类型")
    validate_settings(s)
    directory = new_run(s.output_dir, kind)
    manifest = {"status": "started", "kind": kind, "environment": environment(),
                "settings": asdict(s), "stage": "imports", "limitations":
                "少量固定样本更新，仅验证接线；非能力评估、非QLoRA、非精确断点续训"}
    json_write(directory / "manifest.json", manifest)
    try:
        hf = require_hf()
        device, dtype = resolve_runtime(s)
        require(kind != "local_peft" or (device.type == "cpu" and dtype == torch.float32),
                "--local-peft固定CPU/FP32，用于依赖核对，GPU测试请用--run")
        # 缩小模型不访问Hub：由官方Qwen3Config和模型类随机构建。
        if kind == "local_peft":
            config = small_qwen_config(hf)
            batch = fixture_batch()
            tokenizer = None
            base_id, revision = "random-local-qwen3-not-pretrained", None
            manifest["stage"] = "local_qwen_init"
            with torch.random.fork_rng(devices=[]):
                torch.manual_seed(SEED)
                base = hf.Qwen3ForCausalLM(config)
            local_initial = {n: t.detach().cpu().clone() for n, t in base.state_dict().items()}
            base.to(device)
            base.config.use_cache = False
            plan = plan_from_config(config.to_dict(), s.rank)
            json_write(directory / "data.json", {k: v.tolist() for k, v in batch.items()})
            base.config.save_pretrained(str(directory / "local_base"))
            base.save_pretrained(str(directory / "local_base"), safe_serialization=True)
        else:
            manifest["stage"] = "tokenizer"
            config, tokenizer, batch, info = pretrained_inspection(hf, s)
            json_write(directory / "inspection.json", info)
            base_id, revision, plan = MODEL_ID, REVISION, info["expected_targets"]
            manifest["stage"] = "base_load"
            base = load_real_base(hf, s, device, dtype)
            local_initial = None
        batch = {k: v.to(device) for k, v in batch.items()}
        inventory = discover_targets(base, s.rank)
        verify_inventory(inventory, plan)
        original_refs = list(base.named_parameters())
        original_hash = hash_named_parameters(original_refs)
        baseline = eval_logits(base, batch, device, dtype)
        torch.manual_seed(SEED + 1)
        manifest["stage"] = "peft_injection_and_diagnostics"
        model = inject_with_peft(hf, base, s)
        model.peft_config["default"].base_model_name_or_path = base_id
        model.peft_config["default"].revision = revision
        if s.gradient_checkpointing:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        report, expected_logits = diagnose(model, batch, inventory, s, original_refs,
                                          original_hash, baseline, device, dtype)
        json_write(directory / "audit.json", report)
        adapter_dir = directory / "adapter"
        manifest["stage"] = "save_adapter"
        saved = save_adapter_checked(model, adapter_dir)
        require(saved["tensor_parameters"] == report["initial_audit"]["expected_adapter_parameters"],
                "保存的adapter参数量不符")
        contract = {"format": CONTRACT_FORMAT, "kind": kind, "base_model": base_id,
                    "revision": revision, "base_sha256": original_hash, "base_dtype": str(dtype),
                    "targets": inventory, "lora_spec": lora_spec(s),
                    "environment": environment(), "effective_updates": s.steps,
                    "local_base_path": "local_base" if kind == "local_peft" else None}
        json_write(directory / "contract.json", contract)
        if tokenizer is not None:
            tokenizer.save_pretrained(str(directory / "tokenizer"))
        manifest["stage"] = "fresh_base_reload"
        # 清理所有原模型/优化器引用后再加载，避免3060上同时保留两份大基座。
        del model, base, original_refs
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        if kind == "local_peft":
            fresh = hf.Qwen3ForCausalLM(config)
            fresh.load_state_dict(local_initial, strict=True)
            fresh.to(device)
        else:
            fresh = load_real_base(hf, s, device, dtype)
        require(hash_named_parameters(fresh.named_parameters()) == original_hash, "重载基座指纹不一致")
        restored = hf.PeftModel.from_pretrained(fresh, str(adapter_dir), is_trainable=False,
                                               autocast_adapter_dtype=True)
        reload_audit = audit_adapter(restored, inventory, s, trainable=False)
        got = eval_logits(restored, batch, device, dtype)
        close_outputs(expected_logits, got, dtype)
        reloaded = {"inference_is_trainable": False, "trainable_parameters": reload_audit["trainable_parameters"],
                    "logits_max_abs_diff": max_difference(expected_logits, got), "saved": saved}
        json_write(directory / "reload.json", reloaded)
        # 不生成回答，不宣称两个更新提升语言能力；只核对固定张量与labels。
        manifest.update(status="completed", stage="complete", device=str(device), dtype=str(dtype),
                        real_pretrained_weights=(kind == "real"), cuda_executed=(device.type == "cuda"))
        json_write(directory / "manifest.json", manifest)
        print("完成：", directory)
        print("目标层数：", len(inventory), "adapter参数：", report["initial_audit"]["expected_adapter_parameters"])
        print("初始差异：", report["initial_logits_max_diff"], "基座未变：", report["base_unchanged"])
        print("adapter重载差异：", reloaded["logits_max_abs_diff"])
        print("这是接线冒烟实验，不是能力提升证据。")
        return directory
    except Exception as exc:
        manifest.update(status="failed", error_type=type(exc).__name__, error=str(exc))
        json_write(directory / "manifest.json", manifest)
        (directory / "error.txt").write_text(traceback.format_exc(), encoding="utf-8")
        print("未完成；失败阶段与原始异常：", directory, file=sys.stderr)
        raise


def run_inspection(s: Settings) -> Path:
    directory = new_run(s.output_dir, "inspect")
    manifest = {"status": "started", "kind": "config_tokenizer_only", "environment": environment()}
    try:
        hf = require_hf(need_peft=False)
        _, _, _, info = pretrained_inspection(hf, s)
        json_write(directory / "inspection.json", info)
        manifest["status"] = "completed"
        print("只检查配置/分词器，不加载权重：", directory)
        print("规划adapter参数：", info["expected_adapter_parameters"])
    except Exception as exc:
        manifest.update(status="failed", error=str(exc))
        raise
    finally:
        json_write(directory / "manifest.json", manifest)
    return directory

### 阶段一：真实 PEFT＋随机微型 Qwen3，无需下载权重

设置 `RUN_LOCAL_PEFT = True`，程序会构造官方 `Qwen3Config` 与 `Qwen3ForCausalLM`，然后实际调用 PEFT 注入、反传、保存与重载。

**这是库集成测试，不是有语言能力的模型。** 你需要先安装依赖，但不需要联网下载模型文件。

脚本等价命令：

```bash
python lesson_33_peft_lora_injection_and_auditing.py --local-peft
```

运行失败时保留异常，不要把它与离线替身的成功混在一起。

In [15]:
RUN_LOCAL_PEFT = False
LOCAL_SETTINGS = Settings(device="cpu", dtype="float32", steps=2)
if RUN_LOCAL_PEFT:
    LOCAL_RUN = run_library_experiment("local_peft", LOCAL_SETTINGS)
else:
    print("跳过：真实PEFT本地小模型尚未执行。离线替身结果不能代替此步骤。")

跳过：真实PEFT本地小模型尚未执行。离线替身结果不能代替此步骤。


### 阶段二：真实配置与分词器，只检查数据和规划

这里读取与前几课相同的固定模型 commit，核对 Q/V 形状，并打印**实际参与监督的 token 与对应文字**。

```bash
python lesson_33_peft_lora_injection_and_auditing.py --inspect
```

默认样本仅为“请只回复：你好。”→“你好。”，用途是检查连接。这里并未构建训练/验证集。

首次下载配置和分词器；本地缓存完整后可以加 `--local-files-only`。

In [16]:
RUN_REAL_INSPECTION = False
INSPECTION_SETTINGS = Settings(local_files_only=False)
if RUN_REAL_INSPECTION:
    INSPECTION_RUN = run_inspection(INSPECTION_SETTINGS)
else:
    print("跳过：没有读取真实配置或分词器。上方参数预算来自标明的固定配置算例。")

跳过：没有读取真实配置或分词器。上方参数预算来自标明的固定配置算例。


### 阶段三：真实 Qwen3-0.6B＋LoRA，默认只更新两次

```bash
python lesson_33_peft_lora_injection_and_auditing.py --run --device cuda --dtype float16
```

CPU 可改用 `--device cpu --dtype float32`，但真实模型需要比替身大得多的内存。显式请求 CUDA 时不可用就报错，不静默回退。

本课默认 r=8、alpha=16、序列上限128、batch=1、步数2。`--steps 0` 可以只核对注入与初始保存重载；`--gradient-checkpointing` 开启非重入重计算。先短输入检查，不保证6GB/8GB/12GB的任意软件环境均能装下。

更新次数上限20，防止把这个脚本误当成正式训练器。**没有验证集、没有通用能力成绩，也没有量化基座。**

In [17]:
RUN_REAL_MODEL = False
REAL_SETTINGS = Settings(device="cuda", dtype="float16", rank=8, alpha=16,
                         steps=2, max_length=128, local_files_only=False,
                         gradient_checkpointing=False)
if RUN_REAL_MODEL:
    REAL_RUN = run_library_experiment("real", REAL_SETTINGS)
else:
    print("跳过：真实权重、CUDA与真实模型梯度均未执行。没有模拟回答或显存数据。")

跳过：真实权重、CUDA与真实模型梯度均未执行。没有模拟回答或显存数据。


## 7. 保存的是 adapter，不是另一个完整大模型

本课真正执行 PEFT 路径后，调用：

```python
model.save_pretrained(adapter_dir, safe_serialization=True, save_embedding_layers=False)
```

本课没有扩充词表，没有训练输出头，所以只期望 A/B 张量：

```text
adapter/
    adapter_config.json
    adapter_model.safetensors
    README.md             # PEFT生成的adapter模型卡，不改你的项目README
```

恢复推理时，先加载**同一版本、同一精度的基座**，再执行：

```python
restored = PeftModel.from_pretrained(
    fresh_base,
    adapter_dir,
    is_trainable=False,
)
```

要继续训练 adapter，须明确设置 `is_trainable=True`，并重新核对参数集合与优化器。**这仍然不是精确断点续训**：本课没有保存Adam、scaler、随机状态、数据游标，请回顾第26课的区别。

来源：[PEFT检查点格式](https://huggingface.co/docs/peft/v0.18.0/en/developer_guides/checkpoint)、[加载接口](https://huggingface.co/docs/peft/v0.18.0/en/package_reference/peft_model)。

### 7.1 本课的报告目录

每次新建时间戳子目录，不覆盖之前的运行：

```text
lesson_33_outputs/
    real_时间戳/
        manifest.json           # 环境、设置、成功/失败阶段
        inspection.json         # 实际模板、输入、监督边界
        audit.json              # 参数、logits、梯度、基座哈希
        contract.json           # 固定基座来源、精度、配置、哈希
        reload.json             # 重载后的输出比较
        adapter/                # 真正PEFT格式的adapter
        tokenizer/              # 配套tokenizer，不改变词表
```

`--local-peft` 会另存随机小模型基座以便复查，不能当成预训练Qwen。离线默认只写 `offline.json`、`checks.json` 和环境记录，**不会伪造 adapter 文件**。

失败时 `manifest.json` 标记 `failed`，并保留原始错误，不留下假的通过结论。报告包含样本明文，分享前自行去除私人数据。

## 8. 三项独立练习

先完成自己的实现，再看后面的参考答案。每个开关默认关闭，打开后会真正核对，不用 `try/except` 把错误假装成通过。

**A：参数预算。** 根据每个目标的真实输入输出维度与 rank，计算总可训练参数数。不要假设 Q/V 宽度相同。

**B：同位标签。** 验证完整输入的 token 前缀，提示部分使用 `-100`，保留答案和 EOS，拒绝超长输入，不修改原始列表。

**C：优化器。** 在注入后只收集 `requires_grad=True` 的参数。拒绝空集合，并关闭 weight decay，便于观察 A 首步零梯度。

In [18]:
def budget_ex(rows, rank):
    raise NotImplementedError("请根据每层in/out维度累加r*(in+out)")


def labels_ex(prefix_ids, full_ids, max_length):
    raise NotImplementedError("请验证token前缀并返回input_ids、labels、prefix_length")


def optimizer_ex(model, lr=0.001):
    raise NotImplementedError("请只把当前可训练参数交给AdamW")

RUN_EXERCISE_A = False
RUN_EXERCISE_B = False
RUN_EXERCISE_C = False
print("先完成三个函数，再打开对应检查开关。")

先完成三个函数，再打开对应检查开关。


## 9. 参考答案与独立检查器

**建议先停下来完成练习。** 下面提供的答案使用 `answer_*` 名字，不覆盖你的 `*_ex`。检查器也可以单独用于自己的实现。

In [19]:
def answer_budget(rows: list[dict], rank: int) -> int:
    return sum(lora_count(row["in_features"], row["out_features"], rank) for row in rows)


def answer_labels(prefix_ids, full_ids, max_length):
    # 独立展开，便于与练习核对，不依赖待补函数。
    valid_ids(prefix_ids)
    valid_ids(full_ids)
    positive_int(max_length, "max_length")
    n = len(prefix_ids)
    require(0 < n < len(full_ids) <= max_length, "边界非法")
    require(full_ids[:n] == prefix_ids, "不是同一个token前缀")
    labels = full_ids.copy()
    labels[:n] = [-100] * n
    return {"input_ids": full_ids.copy(), "labels": labels, "prefix_length": n}


def answer_optimizer(model: nn.Module, lr=0.001):
    parameters = [p for p in model.parameters() if p.requires_grad]
    require(len(parameters) > 0, "没有可训练参数")
    return torch.optim.AdamW(parameters, lr=lr, weight_decay=0.0)


def check_exercise_a(fn: Callable) -> None:
    rows = plan_from_config(CONFIG_EXAMPLE, rank=8)
    require(fn(rows, 8) == 1146880, "Qwen配置预算错误")
    require(fn(rows[:2], 4) == 20480, "r变化未反映在结果中")
    require(fn([], 8) == 0, "空列表预算应该为0")


def check_exercise_b(fn: Callable) -> None:
    p, f = [1, 3], [1, 3, 4, 2]
    p0, f0 = p.copy(), f.copy()
    row = fn(p, f, 8)
    require(row == {"input_ids": f, "labels": [-100, -100, 4, 2], "prefix_length": 2}, "mask或标签移位错误")
    require(p == p0 and f == f0, "修改了输入列表")
    row["input_ids"][0] = 99
    require(f == f0, "返回值与输入别名共享")
    for prefix, full, length in [([1], [2, 3], 8), ([1], [1], 8), ([1], [1, 2, 3], 2)]:
        try:
            fn(prefix, full, length)
        except (ValueError, TypeError):
            pass
        else:
            raise AssertionError("没有拒绝不合法边界")


def check_exercise_c(fn: Callable) -> None:
    model, _, _ = make_contract_fixture()
    optimizer = fn(model, lr=0.001)
    assert_optimizer_matches(model, optimizer)
    require(all(group["weight_decay"] == 0 for group in optimizer.param_groups), "本课关闭weight decay")
    # 新注入不同对象以后，旧optimizer应被拒绝。
    other, _, _ = make_contract_fixture()
    try:
        assert_optimizer_matches(other, optimizer)
    except ValueError:
        pass
    else:
        raise AssertionError("旧optimizer身份检查失效")

In [20]:
if RUN_EXERCISE_A:
    check_exercise_a(budget_ex)
    print("练习A通过")
if RUN_EXERCISE_B:
    check_exercise_b(labels_ex)
    print("练习B通过")
if RUN_EXERCISE_C:
    check_exercise_c(optimizer_ex)
    print("练习C通过")
if not any([RUN_EXERCISE_A, RUN_EXERCISE_B, RUN_EXERCISE_C]):
    print("独立练习检查尚未开启；这不代表你已完成练习。")

独立练习检查尚未开启；这不代表你已完成练习。


In [21]:
check_exercise_a(answer_budget)
check_exercise_b(answer_labels)
check_exercise_c(answer_optimizer)
print("三个参考实现通过检查。仍需亲手完成独立练习。")

三个参考实现通过检查。仍需亲手完成独立练习。


## 10. 自动检查与结果复查

下面的固定检查覆盖：目标末段匹配、显式head_dim、参数预算、同位标签、EOS与padding、错误模板边界、初始输出、梯度、基座冻结、优化器身份与序列化。

其中明确使用了纯PyTorch替身的断言，只能证明我们的审计工具发现了对应问题，**不能证明真实PEFT版本兼容或真实Qwen在CUDA上能正常训练**。

脚本命令：

```bash
python lesson_33_peft_lora_injection_and_auditing.py --checks-only
python lesson_33_peft_lora_injection_and_auditing.py --doctor
python lesson_33_peft_lora_injection_and_auditing.py
```

最后一条默认也只运行离线检查和示例，不下载。

### 10.1 固定测试定义

可折叠阅读；不依赖普通 `assert` 处理用户输入。测试失败会停止，而不是继续打印“全部通过”。

In [22]:
class CheckBook:
    def __init__(self):
        self.names = []
    def check(self, name: str, condition=True):
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)
    def raises(self, name: str, fn: Callable, types=(ValueError, TypeError)):
        try:
            fn()
        except types:
            self.names.append(name)
        else:
            raise AssertionError("应当报错：" + name)


def run_checks(verbose=True) -> dict:
    t = CheckBook()
    for name in ["q_proj", "model.layers.0.self_attn.q_proj"]:
        t.check("精确末段命中:" + name, matches_target(name, "q_proj"))
    for name in ["notq_proj", "q_proj_backup", "model.q_proj.base_layer", "model.q_projection"]:
        t.check("拒绝伪命中:" + name, not matches_target(name, "q_proj"))
    plan = plan_from_config(CONFIG_EXAMPLE, 8)
    t.check("Qwen配置示例56层", len(plan) == 56)
    t.check("显式head_dim的Q形状", (plan[0]["in_features"], plan[0]["out_features"]) == (1024, 2048))
    t.check("GQA的V形状", (plan[1]["in_features"], plan[1]["out_features"]) == (1024, 1024))
    t.check("真实配置预算", sum(r["lora_parameters"] for r in plan) == 1146880)
    t.check("r=4预算减半", sum(r["lora_parameters"] for r in plan_from_config(CONFIG_EXAMPLE, 4)) == 573440)
    cfg = dict(hidden_size=8, num_attention_heads=2, num_key_value_heads=1, num_hidden_layers=1)
    t.check("head_dim缺失才推导", plan_from_config(cfg)[0]["out_features"] == 8)
    t.raises("非法GQA比例", lambda: plan_from_config(dict(cfg, num_key_value_heads=3)))
    t.raises("无head_dim且D不能整除", lambda: plan_from_config(dict(cfg, hidden_size=9)))
    for r in (0, -1, True):
        t.raises("非法rank:" + str(r), lambda r=r: lora_count(8, 8, r))
    t.raises("空targets", lambda: plan_from_config(CONFIG_EXAMPLE, targets=[]))
    t.raises("重复targets", lambda: plan_from_config(CONFIG_EXAMPLE, targets=["q_proj", "q_proj"]))
    t.raises("不支持target", lambda: plan_from_config(CONFIG_EXAMPLE, targets=["lm_head"]))
    bad = copy.deepcopy(plan)
    bad[0]["out_features"] //= 2
    t.raises("形状审计发现伪head_dim", lambda: verify_inventory(bad, plan))
    t.raises("漏目标", lambda: verify_inventory(plan[:-1], plan))
    t.raises("重复目标", lambda: verify_inventory(plan + [plan[0]], plan))
    row = row_from_verified_prefix([1, 3], [1, 3, 4, 2], 8)
    t.check("labels不预先移位", row["labels"] == [-100, -100, 4, 2])
    source = [1, 3, 4, 2]
    copied = row_from_verified_prefix([1, 3], source, 8)
    copied["input_ids"][0] = 7
    t.check("不修改源tokens", source == [1, 3, 4, 2])
    for name, p, full, length in [("错前缀", [1], [2, 3], 8), ("空前缀", [], [1, 2], 8),
                                   ("无答案", [1], [1], 8), ("超长", [1], [1, 2, 3], 2),
                                   ("bool token", [True], [True, 2], 8)]:
        t.raises(name, lambda p=p, full=full, length=length: row_from_verified_prefix(p, full, length))
    batch = fixture_batch()
    t.check("右pad attention mask", batch["attention_mask"][1].tolist() == [1, 1, 1, 1, 0, 0])
    t.check("padding label -100", batch["labels"][1].tolist() == [-100, -100, 8, 2, -100, -100])
    t.check("EOS仍监督", supervised_positions(batch)[-1]["target_id"] == 2)
    t.check("首回答由前一位置预测", supervised_positions(batch)[0]["logits_position"] == 2)
    t.raises("空batch", lambda: collate_rows([], 0))
    # PAD与EOS同号时，也仅遮掉真正的填充列。
    same_id = collate_rows([row, row_from_verified_prefix([1], [1, 2], 8)], pad_id=2)
    t.check("按位置而非ID屏蔽EOS", same_id["labels"][1].tolist() == [-100, 2, -100, -100])
    badrow = dict(row, labels=[-100, 4, 2, -100])
    t.raises("拒绝预移位的标签", lambda: collate_rows([badrow], 0))
    logits = torch.zeros(1, 4, 8, requires_grad=True)
    labels = torch.tensor([[-100, -100, 4, 2]])
    loss = manual_causal_loss(logits, labels)
    t.check("均匀loss=log词表", math.isclose(float(loss.detach()), math.log(8), rel_tol=1e-6))
    loss.backward()
    t.check("首位置未受监督", bool((logits.grad[:, 0] == 0).all()))
    t.check("最后位置不预测自己", bool((logits.grad[:, -1] == 0).all()))
    t.check("前缀末位置预测回答", bool(logits.grad[0, 1, 4] < 0))
    t.check("EOS有目标梯度", bool(logits.grad[0, 2, 2] < 0))
    t.raises("拒绝全mask", lambda: manual_causal_loss(logits, torch.full_like(labels, -100)))
    t.raises("拒绝错误标签dtype", lambda: manual_causal_loss(logits, labels.float()))
    t.raises("拒绝logits形状不符", lambda: manual_causal_loss(logits[:, :2], labels))
    # tokenizer 协议替身只用于异常分支，不声称执行了Qwen的BPE或chat_template。
    class TokenizerContractFixture:
        eos_token_id = 2
        def apply_chat_template(self, messages, *, tokenize, add_generation_prompt, enable_thinking):
            require(tokenize and enable_thinking is False, "模板标记错误")
            return [1, 3] if add_generation_prompt else [1, 3, 4, 2, 9]
        def decode(self, ids, skip_special_tokens=False):
            if ids == [9]:
                return "\n"
            return "教学替身token:" + repr(ids)
    fake = TokenizerContractFixture()
    encoded, details = encode_one_turn(fake, "示例问题", "示例回答", 8)
    t.check("模板前缀保留EOS去尾换行", encoded["input_ids"] == [1, 3, 4, 2])
    t.check("答案mask从验证前缀开始", encoded["labels"] == [-100, -100, 4, 2])
    t.check("记录剔除的尾空白", details["removed_trailing_whitespace_ids"] == [9])
    t.raises("空回答拒绝", lambda: encode_one_turn(fake, "问", " ", 8))
    t.raises("答案控制标记拒绝", lambda: encode_one_turn(fake, "问", "<|im_end|>", 8))
    t.raises("模板样本超长拒绝", lambda: encode_one_turn(fake, "问", "答", 3))
    class BadPrefix(TokenizerContractFixture):
        def apply_chat_template(self, messages, **kw):
            return [1, 3] if kw["add_generation_prompt"] else [1, 7, 4, 2, 9]
    t.raises("BPE边界不一致拒绝", lambda: encode_one_turn(BadPrefix(), "问", "答", 8))
    class BadTail(TokenizerContractFixture):
        def decode(self, ids, skip_special_tokens=False):
            return "非空尾部"
    t.raises("EOS之后非空内容拒绝", lambda: encode_one_turn(BadTail(), "问", "答", 8))
    class NoEnd(TokenizerContractFixture):
        def apply_chat_template(self, messages, **kw):
            return [1, 3] if kw["add_generation_prompt"] else [1, 3, 4, 5]
    t.raises("缺少EOS拒绝", lambda: encode_one_turn(NoEnd(), "问", "答", 8))
    s = Settings(rank=2, alpha=4, steps=2, lr=0.01)
    model, inventory, refs = make_contract_fixture(s)
    audit = audit_adapter(model, inventory, s, initial=True)
    t.check("替身注入4层", audit["target_modules"] == 4)
    t.check("替身8个adapter张量", audit["adapter_tensor_count"] == 8)
    t.check("替身adapter预算112", audit["trainable_parameters"] == 112)
    t.raises("重复注入拒绝", lambda: discover_targets(model.get_base_model(), 2))
    t.raises("未命中target", lambda: discover_targets(ContractLM(), 2, targets=("missing",)))
    raw = ContractLM()
    raw.model.layers[0].self_attn.q_proj = nn.Identity()
    t.raises("非Linear拒绝", lambda: discover_targets(raw, 2))
    old_hash = hash_named_parameters(refs)
    t.check("分块哈希无关块大小", old_hash == hash_named_parameters(refs, 7))
    t.check("仅adapter可训练", all(not p.requires_grad for _, p in refs))
    refs[0][1].requires_grad_(True)
    t.raises("意外解冻被发现", lambda: audit_adapter(model, inventory, s))
    refs[0][1].requires_grad_(False)
    opt = answer_optimizer(model, 0.01)
    assert_optimizer_matches(model, opt)
    t.check("优化器恰好包含adapter")
    wrong_opt = torch.optim.AdamW([p for _, p in refs], lr=0.01)
    t.raises("旧optimizer拒绝", lambda: assert_optimizer_matches(model, wrong_opt))
    device, dtype = torch.device("cpu"), torch.float32
    with model.disable_adapter():
        baseline = eval_logits(model, batch, device, dtype)
    initial = eval_logits(model, batch, device, dtype)
    t.check("零增量初始logits完全一致", torch.equal(initial, baseline))
    model.train()
    _ = eval_logits(model, batch, device, dtype)
    t.check("eval辅助恢复train", model.training)
    t.check("eval不产生梯度", all(p.grad is None for p in model.parameters()))
    before = snapshot_trainable(model)
    scaler = torch.amp.GradScaler("cpu", enabled=False)
    first = one_update(model, batch, inventory, opt, scaler, device, dtype)
    t.check("首步A梯度为零", first["gradients"]["A_grad_norm"] == 0)
    t.check("首步B梯度非零", first["gradients"]["B_grad_norm"] > 0)
    t.check("所有adapter都有grad记录", not first["gradients"]["missing"])
    t.check("未切断冻结层反传", not first["gradients"]["nonfinite"])
    changes = parameter_changes(model, before)
    t.check("adapter变化", changes["changed_tensors"] > 0)
    t.check("基座无grad", all(p.grad is None for _, p in refs))
    second = one_update(model, batch, inventory, opt, scaler, device, dtype)
    t.check("第二步A梯度非零", second["gradients"]["A_grad_norm"] > 0)
    t.check("两步后基座哈希不变", hash_named_parameters(refs) == old_hash)
    adapted = eval_logits(model, batch, device, dtype)
    t.check("适配后logits变化", max_difference(adapted, baseline) > 0)
    with model.disable_adapter():
        disabled = eval_logits(model, batch, device, dtype)
    t.check("关闭adapter回到基座", torch.equal(disabled, baseline))
    t.check("关闭上下文后重新生效", torch.equal(eval_logits(model, batch, device, dtype), adapted))
    # 依赖无关的序列化核对，只称为torch state_dict，不称为PEFT checkpoint。
    with tempfile.TemporaryDirectory() as temp:
        path = Path(temp) / "fixture.pt"
        torch.save(model.state_dict(), path)
        restored, _, _ = make_contract_fixture(s)
        restored.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)
        t.check("torch替身重载一致", torch.equal(eval_logits(restored, batch, device, dtype), adapted))
    bad_s = Settings(rank=True)
    t.raises("配置拒绝bool", lambda: validate_settings(bad_s))
    t.raises("过多步骤不是本课任务", lambda: validate_settings(Settings(steps=21)))
    t.raises("负lr", lambda: validate_settings(Settings(lr=-1)))
    t.raises("CPU半精度不冒充支持", lambda: resolve_runtime(Settings(dtype="float16")))
    for name, check, answer in [("A", check_exercise_a, answer_budget), ("B", check_exercise_b, answer_labels),
                                ("C", check_exercise_c, answer_optimizer)]:
        check(answer)
        t.check("练习" + name + "参考实现")
    result = {"passed": len(t.names), "names": t.names,
              "scope": "标准库+PyTorch离线数学/协议/替身；没有执行真实PEFT或Qwen权重"}
    if verbose:
        print(f"离线检查通过：{len(t.names)} 项。不是 PEFT/Qwen/CUDA 实测。")
    return result

In [23]:
CHECK_REPORT = run_checks()
print(CHECK_REPORT["scope"])
print("实际检查数量：", CHECK_REPORT["passed"])
print("真实PEFT本地阶段：", "本次由用户开启" if RUN_LOCAL_PEFT else "未执行")
print("真实Qwen权重阶段：", "本次由用户开启" if RUN_REAL_MODEL else "未执行")

离线检查通过：87 项。不是 PEFT/Qwen/CUDA 实测。
标准库+PyTorch离线数学/协议/替身；没有执行真实PEFT或Qwen权重
实际检查数量： 87
真实PEFT本地阶段： 未执行
真实Qwen权重阶段： 未执行


## 11. 口头验收：能否解释这些现象？

1. 为什么 `print_trainable_parameters()` 数字正确，optimizer 仍然可能没有更新 LoRA？
2. 为什么 A 首步梯度为0，但这不是训练失败？`None` 和0又有什么不同？
3. 为什么 loss 屏蔽用户提示后，模型仍可利用用户提示？
4. 为什么这次不能预先把 labels 向左移一位？
5. 为什么两次变量名不同的 `base` / `model` 不一定是两个独立模型？
6. 为什么 adapter 文件很小，训练峰值显存仍可能很大？
7. 为什么同样大小的基座或相同参数形状还不够，重载要固定来源和版本？
8. 为什么两步训练后loss降低，不足以证明能力提升？

**通过标准：** 你能逐项解释审计报告；真实依赖阶段成功后，保存的adapter在干净基座上重载，固定输入的logits在容差内一致；基座字节哈希不变。

不要急着扩大数据或训练步数，先把“只更新谁、监督什么、怎样恢复”核对清楚。

## 12. 边界、常见故障与下一课

| 现象 | 首先检查 |
|---|---|
| 层名找不到 | 是否把其他模型的名字照抄过来？是否已经重复注入？ |
| 没有可训练参数 | 是否恢复时使用了默认的 `is_trainable=False`？ |
| B没有梯度 | 标签是否全部忽略？前向是否被 `no_grad()` 包住？优化目标是否连接到这层？ |
| loss为NaN | 有效标签数是否为0？梯度/前向是否溢出？不要把无有效目标的mean当正常值 |
| 初始输出不一致 | 是否启用了随机Dropout、非默认初始化、不同dtype或不同输入？ |
| 显存不足 | 当前是否残留两份模型？先batch=1、短序列；可用非重入检查点；量化训练另讲 |
| adapter重载不同 | 基座版本、dtype、目标与adapter状态是否配套？ |

本课没有 Trainer、没有QLoRA、没有多卡、没有完整SFT训练和评测，没有实际GPU资源基准。它建立的是**真实库的可验证接线流程**。

**下一课：第34课——监督微调数据、Assistant-only Loss 与多轮对话。** 在本课单轮短样本的基础上，系统处理数据划分、多轮模板、损失掩码、截断与批处理，再进入正式的小模型SFT。

## 一手资料

- [PEFT LoRA 参数与目标匹配](https://huggingface.co/docs/peft/v0.18.0/package_reference/lora)
- [PEFT 模型、adapter dtype、disable_adapter、is_trainable](https://huggingface.co/docs/peft/v0.18.0/en/package_reference/peft_model)
- [PEFT 就地注入与优化器创建前的对象变化](https://huggingface.co/docs/peft/guides/peft_model_config)
- [PEFT adapter 检查点格式](https://huggingface.co/docs/peft/v0.18.0/en/developer_guides/checkpoint)
- [所用固定 Qwen3 配置](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/config.json)
- [所用固定 Qwen3 模板](https://huggingface.co/Qwen/Qwen3-0.6B/raw/c1899de289a04d12100db370d81485cdf75e47ca/tokenizer_config.json)
- [Transformers 4.57.3 Qwen3 实现](https://raw.githubusercontent.com/huggingface/transformers/v4.57.3/src/transformers/models/qwen3/modeling_qwen3.py)
- [Transformers 4.57.3 CausalLM loss 的内部移位](https://raw.githubusercontent.com/huggingface/transformers/v4.57.3/src/transformers/loss/loss_utils.py)
- [PyTorch AMP：反传、去缩放与裁剪](https://docs.pytorch.org/docs/stable/notes/amp_examples.html)

这些来源用于说明API和计算约定；本讲义的数值示例以实际运行输出为准，缺失的真实实验不填造结果。